In [ ]:
!pip install qai-hub

In [ ]:
!qai-hub configure --api_token jbv2p4zosz3wgkuaczkoqw940rbr9ihmjqutoa4p

2026-05-18 06:29:13.573 - INFO - Enabling verbose logging.
qai-hub configuration saved to /root/.qai_hub/client.ini
==================== /root/.qai_hub/client.ini ====================
[api]
api_token = jbv2p4zosz3wgkuaczkoqw940rbr9ihmjqutoa4p
api_url = https://workbench.aihub.qualcomm.com
web_url = https://workbench.aihub.qualcomm.com
verbose = True
client_mode = cli




In [ ]:
!qai-hub list-devices

+---------------------------------+--------------+----------+---------+---------------------------------------------------+------------------------------------------------------------+
|              Device             |      OS      |  Vendor  |   Type  |                      Chipset                      |                       CLI Invocation                       |
+---------------------------------+--------------+----------+---------+---------------------------------------------------+------------------------------------------------------------+
|     Google Pixel 3 (Family)     |  Android 10  |  Google  |  Phone  |          qualcomm-snapdragon-845, sdm845          |     --device "Google Pixel 3 (Family)" --device-os 10      |
|          Google Pixel 3         |  Android 10  |  Google  |  Phone  |          qualcomm-snapdragon-845, sdm845          |          --device "Google Pixel 3" --device-os 10          |
|         Google Pixel 3a         |  Android 10  |  Google  |  Phone  |    

In [ ]:
!pip3 install qai-hub-models

In [ ]:
!python -m qai_hub_models.models.efficientnet_b0.demo

Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth
100% 20.5M/20.5M [00:00<00:00, 47.9MB/s]
Model Loaded
Top 5 predictions for image:

Samoyed: 84.7%

Pomeranian: 1.86%

Arctic fox: 1.08%

husky: 0.679%

Keeshond: 0.507%



In [ ]:
!pip3 install 'qai-hub[torch]'

In [ ]:
# Import torch and pre-trained MobileNet
import torch
from torchvision.models import efficientnet_b0

# Load the pre-trained model
torch_model = efficientnet_b0(pretrained=True)
torch_model.eval()

# Trace model (for on-device deployment)
input_shape = (1, 3, 224, 224)
example_input = torch.rand(input_shape)
traced_torch_model = torch.jit.trace(torch_model, example_input)

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=EfficientNet_B0_Weights.IMAGENET1K_V1`. You can also use `weights=EfficientNet_B0_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


In [ ]:
import qai_hub as hub
device = hub.Device("Samsung Galaxy S25 (Family)")

# Optimize model for the chosen device using the NPU
compile_job = hub.submit_compile_job(
        model=traced_torch_model,
        device=device,
        input_specs=dict(image=input_shape),
        options="--target_runtime tflite --compute_unit npu"
)
target_model = compile_job.get_target_model()
print(target_model)

Uploading tmp8o0kh_3_.pt


100%|██████████| 21.0M/21.0M [00:01<00:00, 18.8MB/s]


Scheduled compile job (jgjkryk85) successfully. To see the status and results:
    https://workbench.aihub.qualcomm.com/jobs/jgjkryk85/

Waiting for compile job (jgjkryk85) completion. Type Ctrl+C to stop waiting at any time.
    ✅ SUCCESS                          
Model(model_id='mnlej7rkq', name='job_jgjkryk85_optimized_tflite')


### Targeting Dragonwing RB3 Gen 2 Vision Kit
In this section, we configure the hardware for your specific IoT kit. Preprocessing (like image resizing) remains on the CPU, while the inference is offloaded to the NPU.

In [ ]:
!pip install qai_hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 122.0/122.0 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.3/85.3 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.0/15.0 MB 58.1 MB/s eta 0:00:00


In [ ]:
!qai-hub configure --api_token jbv2p4zosz3wgkuaczkoqw940rbr9ihmjqutoa4p

2026-05-19 10:11:35.684 - INFO - Enabling verbose logging.
qai-hub configuration saved to /root/.qai_hub/client.ini
==================== /root/.qai_hub/client.ini ====================
[api]
api_token = jbv2p4zosz3wgkuaczkoqw940rbr9ihmjqutoa4p
api_url = https://workbench.aihub.qualcomm.com
web_url = https://workbench.aihub.qualcomm.com
verbose = True
client_mode = cli




Quantization

In [ ]:
!pip install numpy
!pip install pillow

In [ ]:
import qai_hub as hub
import os
import shutil
import glob
import numpy as np
from PIL import Image

# Define the RB3 Gen 2 Vision Kit device
rb3_device = hub.Device("Dragonwing RB3 Gen 2 Vision Kit")

# --- Steps to upload and compile your ONNX model ---

# Prepare a CLEAN directory for upload
export_dir = "/content/model_upload"
os.makedirs(export_dir, exist_ok=True)
onnx_file = "drone_classifier.onnx"
data_file = "drone_classifier.onnx.data"

if os.path.exists(f"/content/{onnx_file}"):
    shutil.copy2(f"/content/{onnx_file}", f"{export_dir}/{onnx_file}")
if os.path.exists(f"/content/{data_file}"):
    shutil.copy2(f"/content/{data_file}", f"{export_dir}/{data_file}")

input_shape = (1, 3, 224, 224)

# Upload using the clean directory
onnx_source_model = hub.upload_model(
    model=export_dir,
    name="Drone_Classifier_ONNX_Fix"
)

# Quantization
calibration_folder = "/content/drive/MyDrive/DroneDetectionCollab/raw_inputs"
image_extensions = ('*.png', '*.jpg', '*.jpeg')
calibration_files = []
for ext in image_extensions:
    calibration_files.extend(glob.glob(os.path.join(calibration_folder, ext)))

if len(calibration_files) == 0:
    raise FileNotFoundError(f"No images found in {calibration_folder}. Please check your path.")

print(f"Found {len(calibration_files)} images for calibration. Processing...")

processed_calibration_images = []

for img_path in calibration_files[:50]:
    try:
        img = Image.open(img_path).convert("RGB").resize((224, 224))
        img_array = np.transpose(np.array(img, dtype=np.float32) / 255.0, (2, 0, 1))

        # CRITICAL SHAPE FIX: Add the batch dimension back so each entry is (1, 3, 224, 224)
        img_array_4d = np.expand_dims(img_array, axis=0)
        processed_calibration_images.append(img_array_4d)
    except Exception as e:
        print(f"Skipping unreadable image {img_path}: {e}")

if len(processed_calibration_images) == 0:
    raise ValueError("Dataset construction aborted: 0 images were successfully processed.")

# Construct the calibration dataset dictionary matching your actual model input layer name
cal_data = {"roi_patch": processed_calibration_images}
print(f"Successfully processed {len(processed_calibration_images)} images into 4D calibration arrays.")

print("\nSubmitting Quantization Job (FP32 -> INT8) using real calibration data...")
quantize_job = hub.submit_quantize_job(
    model=onnx_source_model,
    calibration_data=cal_data,
    weights_dtype=hub.QuantizeDtype.INT8,
    activations_dtype=hub.QuantizeDtype.INT8,
)
quantized_model = quantize_job.get_target_model()



Uploading model_upload.zip


100%|██████████| 26.0M/26.0M [00:01<00:00, 22.3MB/s]


Found 100 images for calibration. Processing...
Successfully processed 50 images into 4D calibration arrays.

Submitting Quantization Job (FP32 -> INT8) using real calibration data...


Uploading dataset: 5.02MB [00:00, 5.77MB/s]                            


Scheduled quantize job (jgkr2woo5) successfully. To see the status and results:
    https://workbench.aihub.qualcomm.com/jobs/jgkr2woo5/

Waiting for quantize job (jgkr2woo5) completion. Type Ctrl+C to stop waiting at any time.
    ✅ SUCCESS                          


In [ ]:
print(f"Submitting compilation job for {rb3_device.name}...")
rb3_compile_job = hub.submit_compile_job(
    model=quantized_model,
    device=rb3_device,
    input_specs=dict(roi_patch=input_shape),
    options="--target_runtime tflite --compute_unit npu"
)

rb3_model = rb3_compile_job.get_target_model()
print(f"Model compiled successfully for {rb3_device.name} NPU.\n")

Submitting compilation job for Dragonwing RB3 Gen 2 Vision Kit...
Scheduled compile job (jp23rqe4g) successfully. To see the status and results:
    https://workbench.aihub.qualcomm.com/jobs/jp23rqe4g/

Waiting for compile job (jp23rqe4g) completion. Type Ctrl+C to stop waiting at any time.
    ✅ SUCCESS                          
Model compiled successfully for Dragonwing RB3 Gen 2 Vision Kit NPU.



In [ ]:
# Inference
image_path = "/content/dji_cao50_xa50_low__seg0010_start700.0ms.png"
if os.path.exists(image_path):
    image = Image.open(image_path).convert("RGB").resize((224, 224))
    input_array = np.transpose(np.array(image, dtype=np.float32) / 255.0, (2, 0, 1))
    input_array = np.expand_dims(input_array, axis=0)

    print(f"Submitting inference job for {image_path}...")
    rb3_inference_job = hub.submit_inference_job(
        model=rb3_model,
        device=rb3_device,
        inputs=dict(roi_patch=[input_array]),
        options="--compute_unit npu"
    )

    rb3_output = rb3_inference_job.download_output_data()
    print("Inference successful.")
    display(rb3_output)

Submitting inference job for /content/dji_cao50_xa50_low__seg0010_start700.0ms.png...


Uploading dataset: 111kB [00:00, 195kB/s]                            


Scheduled inference job (jpvzlynkg) successfully. To see the status and results:
    https://workbench.aihub.qualcomm.com/jobs/jpvzlynkg/

Waiting for inference job (jpvzlynkg) completion. Type Ctrl+C to stop waiting at any time.
    ✅ SUCCESS                          


tmp_tt_606x.h5: 100%|██████████| 13.9k/13.9k [00:00<00:00, 3.83MB/s]

Inference successful.


{'output_0': [array([[-1.5958469 ,  1.9504795 ,  0.02216454, -0.17731632, -1.0417334 ,
          -1.8174924 ,  1.1525561 ,  2.0169733 ]], dtype=float32)]}

In [ ]:
print(f"\nSubmitting profile job to {rb3_device.name}...")
rb3_profile_job = hub.submit_profile_job(
    model=rb3_model,
    device=rb3_device,
    options="--compute_unit npu"
)

profile_results = rb3_profile_job.download_profile()
latency = profile_results['execution_summary']['estimated_inference_time']
memory = profile_results['execution_summary']['estimated_inference_peak_memory'] / (1024 * 1024)

print("-" * 50)
print(f"Performance Summary (True NPU INT8) for {rb3_device.name}:")
print(f"Avg Latency: {latency:.2f} ms")
print(f"Peak Memory: {memory:.2f} MB")
print("-" * 50)


Submitting profile job to Dragonwing RB3 Gen 2 Vision Kit...
Scheduled profile job (jg99z7e8g) successfully. To see the status and results:
    https://workbench.aihub.qualcomm.com/jobs/jg99z7e8g/

Waiting for profile job (jg99z7e8g) completion. Type Ctrl+C to stop waiting at any time.
    ✅ SUCCESS                          
--------------------------------------------------
Performance Summary (True NPU INT8) for Dragonwing RB3 Gen 2 Vision Kit:
Avg Latency: 1575.00 ms
Peak Memory: 17.68 MB
--------------------------------------------------


In [ ]:
import qai_hub as hub

# Retrieve the existing optimized target model asset using its ID from yesterday
rb3_model = hub.get_model("mnzlyx9zm")

# Download the final .tflite binary file to your Colab /content/ folder
rb3_model.download("drone_classifier_quantized.tflite")
print("✓ Model successfully retrieved and downloaded as drone_classifier_quantized.tflite")

drone_classifier_quantized.tflite: 100%|██████████| 7.06M/7.06M [00:00<00:00, 41.8MB/s]

Downloaded model to drone_classifier_quantized.tflite
✓ Model successfully retrieved and downloaded as drone_classifier_quantized.tflite


In [ ]:
import qai_hub as hub

# Fetch the generalized jobs list from your account
jobs = hub.get_jobs()

print("Recent Jobs History:")
for job in jobs[:10]:  # Inspect the last 10 jobs
    # Only print compile or quantize jobs to keep it clean
    if "compile" in job.job_id or "quantize" in job.job_id:
        print(f"Job ID: {job.job_id} | Status: {job.get_status().state}")

AttributeError: module 'qai_hub' has no attribute 'get_jobs'